In [1]:
from fast_borf.weighted.iborf import IBORF
import xarray as xr
import numpy as np

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import f1_score

In [20]:
from irregular_ts.data_utils import data_new_folder
import xarray as xr
df = xr.open_dataset(data_new_folder() / "Abf.h5", engine="my_engine")["data"]
y, split = df.irr.get_task_target_and_split()
X, _ = df.irr.to_dense(
    concatenate_time=True,
    normalize_time=True,
)
train_idxs, test_idxs = split == "train", split == "test"
X_train, y_train = X[train_idxs], y[train_idxs]
X_test, y_test = X[test_idxs], y[test_idxs]
X_train.shape

(30, 2, 128)

In [21]:
borf = IBORF(
    contains_time_idx=True,
    min_window_to_signal_std_ratio=0.15,
    n_jobs=-1
)

In [22]:
pipe = make_pipeline(
    borf,
    FunctionTransformer(lambda x: np.arcsinh(x)),
    RidgeClassifierCV()
)

In [23]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
f1_score(y_test, y_pred, average="macro")

0.8951712319329848